# 09 — Symbolic PPO: Train on World 1-1

Trains a PPO agent on **World 1-1** using the RAM-based symbolic grid (13×16).
The saved model is the prerequisite for **notebook 10** (multi-task training).

**Setup:**
- Observation: flattened 13×16 grid × 4 frame-stack = **832-dim** vector
- Policy: `MlpPolicy` with `[512, 512]` hidden layers
- 8 parallel envs, two training phases (2 M + 2 M steps)
- Saves to `models/symbolic_ppo/final_model`

In [1]:
# --- Google Colab Setup ---
import os, sys

if os.getenv('COLAB_RELEASE_TAG'):
    !pip install -q Cython setuptools wheel
    !git clone -b hotfix/version1 https://github.com/lmartim4/CSC-52081-ContinousMountainCar.git /content/repo
    %cd /content/repo
    !pip install -r requirements.txt --no-build-isolation
    sys.path.insert(0, '/content/repo')
    import site; SITE = site.getsitepackages()[0]
    !patch --forward -p0 {SITE}/nes_py/_rom.py                   < patches/nes_py_numpy2.patch || true
    !patch --forward -p0 {SITE}/gym/utils/passive_env_checker.py < patches/gym_bool8_numpy2.patch || true
    !patch --forward -p0 {SITE}/gym_super_mario_bros/smb_env.py  < patches/smb_env_numpy2.patch || true
    !sed -i 's/observation, reward, terminated, truncated, info = self.env.step(action)/_result = self.env.step(action); observation, reward, terminated, info = _result[:4]; truncated = _result[4] if len(_result) > 4 else False/' {SITE}/gym/wrappers/time_limit.py
    !git pull
else:
    if os.path.basename(os.getcwd()) == 'notebooks':
        %cd ..

/home/lmartim4/Desktop/CSC-52081-Mario


In [2]:
import torch
from stable_baselines3 import PPO

from src.wrappers import make_symbolic_vec_env, make_symbolic_env
from src.utils.callbacks import CheckpointAndLogCallback
from src.config import PPOConfig

print(f'Using CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

Using CUDA: True
GPU: NVIDIA GeForce RTX 3060 Ti


Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


In [3]:
# Create 8 parallel symbolic environments for World 1-1
NUM_ENVS = 8

env = make_symbolic_vec_env(
    env_id='SuperMarioBros-1-1-v3',
    skip=4,
    n_stack=4,
    flatten=True,
    num_envs=NUM_ENVS,
)
print(f'Observation space: {env.observation_space.shape}')
print(f'Action space: {env.action_space.n}')
print(f'Parallel envs: {NUM_ENVS}')

Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.
Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


Observation space: (832,)
Action space: 7
Parallel envs: 8


/home/lmartim4/Desktop/CSC-52081-Mario/.venv/lib/python3.14/site-packages/stable_baselines3/common/vec_env/patch_gym.py:49: UserWarning: You provided an OpenAI Gym environment. We strongly recommend transitioning to Gymnasium environments. Stable-Baselines3 is automatically wrapping your environments in a compatibility layer, which could potentially cause issues.
  warnings.warn(
/home/lmartim4/Desktop/CSC-52081-Mario/.venv/lib/python3.14/site-packages/stable_baselines3/common/vec_env/patch_gym.py:49: UserWarning: You provided an OpenAI Gym environment. We strongly recommend transitioning to Gymnasium environments. Stable-Baselines3 is automatically wrapping your environments in a compatibility layer, which could potentially cause issues.
  warnings.warn(
/home/lmartim4/Desktop/CSC-52081-Mario/.venv/lib/python3.14/site-packages/stable_baselines3/common/vec_env/patch_gym.py:49: UserWarning: You provided an OpenAI Gym environment. We strongly recommend transitioning to Gymnasium environm

In [4]:
# Phase 1: Train from scratch — 2 M steps
config = PPOConfig(
    policy='MlpPolicy',
    learning_rate=2.5e-4,
    n_steps=512,
    batch_size=256,
    n_epochs=4,
    gamma=0.9,
    gae_lambda=0.95,
    clip_range=0.2,
    ent_coef=0.01,
    vf_coef=0.5,
    max_grad_norm=0.5,
    total_timesteps=2_000_000,
)

model = PPO(
    policy=config.policy,
    env=env,
    learning_rate=config.learning_rate,
    n_steps=config.n_steps,
    batch_size=config.batch_size,
    n_epochs=config.n_epochs,
    gamma=config.gamma,
    gae_lambda=config.gae_lambda,
    clip_range=config.clip_range,
    ent_coef=config.ent_coef,
    vf_coef=config.vf_coef,
    max_grad_norm=config.max_grad_norm,
    policy_kwargs=dict(net_arch=[512, 512]),
    tensorboard_log='../logs/symbolic_ppo',
    verbose=1,
    device='cpu',  # MLP is faster on CPU
)

print(f'Phase 1: {config.total_timesteps:,} timesteps')
print(f'Device: {model.device}')
print(f'Batch per rollout: {config.n_steps * NUM_ENVS} samples')

Using cpu device
Phase 1: 2,000,000 timesteps
Device: cpu
Batch per rollout: 4096 samples


In [5]:
%load_ext tensorboard
%tensorboard --logdir ../logs/symbolic_ppo

ERROR: Failed to launch TensorBoard (exited with 1).
Contents of stderr:
Traceback (most recent call last):
  File "/home/lmartim4/Desktop/CSC-52081-Mario/.venv/bin/tensorboard", line 3, in <module>
    from tensorboard.main import run_main
  File "/home/lmartim4/Desktop/CSC-52081-Mario/.venv/lib/python3.14/site-packages/tensorboard/main.py", line 27, in <module>
    from tensorboard import default
  File "/home/lmartim4/Desktop/CSC-52081-Mario/.venv/lib/python3.14/site-packages/tensorboard/default.py", line 30, in <module>
    import pkg_resources
ModuleNotFoundError: No module named 'pkg_resources'

In [6]:
callback = CheckpointAndLogCallback(
    save_path='../models/symbolic_ppo',
    save_freq=50_000,
)

model.learn(
    total_timesteps=config.total_timesteps,
    callback=callback,
    log_interval=1,
)
model.save('../models/symbolic_ppo/phase1_model')
print('Phase 1 complete!')

Logging to ../logs/symbolic_ppo/PPO_1


/home/lmartim4/Desktop/CSC-52081-Mario/.venv/lib/python3.14/site-packages/gym/utils/passive_env_checker.py:195: UserWarning: WARN: The result returned by `env.reset()` was not a tuple of the form `(obs, info)`, where `obs` is a observation and `info` is a dictionary containing additional information. Actual type: `<class 'numpy.ndarray'>`
  logger.warn(
/home/lmartim4/Desktop/CSC-52081-Mario/.venv/lib/python3.14/site-packages/gym/utils/passive_env_checker.py:195: UserWarning: WARN: The result returned by `env.reset()` was not a tuple of the form `(obs, info)`, where `obs` is a observation and `info` is a dictionary containing additional information. Actual type: `<class 'numpy.ndarray'>`
  logger.warn(
/home/lmartim4/Desktop/CSC-52081-Mario/.venv/lib/python3.14/site-packages/gym/utils/passive_env_checker.py:195: UserWarning: WARN: The result returned by `env.reset()` was not a tuple of the form `(obs, info)`, where `obs` is a observation and `info` is a dictionary containing additional

-----------------------------
| time/              |      |
|    fps             | 570  |
|    iterations      | 1    |
|    time_elapsed    | 7    |
|    total_timesteps | 4096 |
-----------------------------
-----------------------------------------
| rollout/                |             |
|    ep_len_mean          | 237         |
|    ep_rew_mean          | 56          |
|    flag_rate_100        | 0           |
| time/                   |             |
|    fps                  | 552         |
|    iterations           | 2           |
|    time_elapsed         | 14          |
|    total_timesteps      | 8192        |
| train/                  |             |
|    approx_kl            | 0.010723841 |
|    clip_fraction        | 0.111       |
|    clip_range           | 0.2         |
|    entropy_loss         | -1.94       |
|    explained_variance   | -0.047      |
|    learning_rate        | 0.00025     |
|    loss                 | 0.78        |
|    n_updates            | 4     

/home/lmartim4/Desktop/CSC-52081-Mario/.venv/lib/python3.14/site-packages/gym_super_mario_bros/smb_env.py:177: RuntimeWarning: overflow encountered in scalar add
  return 255 + (255 - self._y_pixel)


-----------------------------------------
| rollout/                |             |
|    ep_len_mean          | 156         |
|    ep_rew_mean          | 98.3        |
|    flag_rate_100        | 0.01        |
| time/                   |             |
|    fps                  | 515         |
|    iterations           | 26          |
|    time_elapsed         | 206         |
|    total_timesteps      | 106496      |
| train/                  |             |
|    approx_kl            | 0.011389565 |
|    clip_fraction        | 0.122       |
|    clip_range           | 0.2         |
|    entropy_loss         | -0.954      |
|    explained_variance   | 0.515       |
|    learning_rate        | 0.00025     |
|    loss                 | 1.11        |
|    n_updates            | 100         |
|    policy_gradient_loss | -0.0108     |
|    value_loss           | 3.35        |
-----------------------------------------


...

-----------------------------------------
| rollout/                |             |
|    ep_len_mean          | 260         |
|    ep_rew_mean          | 191         |
|    flag_rate_100        | 0.21        |
| time/                   |             |
|    fps                  | 514         |
|    iterations           | 61          |
|    time_elapsed         | 486         |
|    total_timesteps      | 249856      |
| train/                  |             |
|    approx_kl            | 0.018002369 |
|    clip_fraction        | 0.184       |
|    clip_range           | 0.2         |
|    entropy_loss         | -0.842      |
|    explained_variance   | 0.751       |
|    learning_rate        | 0.00025     |
|    loss                 | 0.797       |
|    n_updates            | 240         |
|    policy_gradient_loss | -0.00927    |
|    value_loss           | 1.73        |
-----------------------------------------
-----------------------------------------
| time/                   |       

/home/lmartim4/Desktop/CSC-52081-Mario/.venv/lib/python3.14/site-packages/gym_super_mario_bros/smb_env.py:177: RuntimeWarning: overflow encountered in scalar add
  return 255 + (255 - self._y_pixel)


-----------------------------------------
| time/                   |             |
|    fps                  | 514         |
|    iterations           | 64          |
|    time_elapsed         | 509         |
|    total_timesteps      | 262144      |
| train/                  |             |
|    approx_kl            | 0.013820053 |
|    clip_fraction        | 0.159       |
|    clip_range           | 0.2         |
|    entropy_loss         | -0.798      |
|    explained_variance   | 0.775       |
|    learning_rate        | 0.00025     |
|    loss                 | 0.527       |
|    n_updates            | 252         |
|    policy_gradient_loss | -0.0102     |
|    value_loss           | 1.67        |
-----------------------------------------
-----------------------------------------
| rollout/                |             |
|    ep_len_mean          | 293         |
|    ep_rew_mean          | 213         |
|    flag_rate_100        | 0.31        |
| time/                   |       

/home/lmartim4/Desktop/CSC-52081-Mario/.venv/lib/python3.14/site-packages/gym_super_mario_bros/smb_env.py:177: RuntimeWarning: overflow encountered in scalar add
  return 255 + (255 - self._y_pixel)


-----------------------------------------
| rollout/                |             |
|    ep_len_mean          | 296         |
|    ep_rew_mean          | 217         |
|    flag_rate_100        | 0.3         |
| time/                   |             |
|    fps                  | 514         |
|    iterations           | 67          |
|    time_elapsed         | 533         |
|    total_timesteps      | 274432      |
| train/                  |             |
|    approx_kl            | 0.018666014 |
|    clip_fraction        | 0.149       |
|    clip_range           | 0.2         |
|    entropy_loss         | -0.786      |
|    explained_variance   | 0.771       |
|    learning_rate        | 0.00025     |
|    loss                 | 0.601       |
|    n_updates            | 264         |
|    policy_gradient_loss | -0.00924    |
|    value_loss           | 1.62        |
-----------------------------------------
----------------------------------------
| time/                   |        

In [7]:
# Phase 2: Fine-tune with lower LR — 2 M more steps
model2 = PPO.load('../models/symbolic_ppo/phase1_model', env=env, device='cpu')
model2.learning_rate = 1e-5

callback2 = CheckpointAndLogCallback(
    save_path='../models/symbolic_ppo',
    save_freq=50_000,
)

model2.learn(
    total_timesteps=2_000_000,
    callback=callback2,
    log_interval=1,
)
model2.save('../models/symbolic_ppo/final_model')
print('Phase 2 complete — final_model saved!')

Logging to ../logs/symbolic_ppo/PPO_2
-----------------------------
| time/              |      |
|    fps             | 534  |
|    iterations      | 1    |
|    time_elapsed    | 7    |
|    total_timesteps | 4096 |
-----------------------------
-----------------------------------------
| rollout/                |             |
|    ep_len_mean          | 313         |
|    ep_rew_mean          | 293         |
|    flag_rate_100        | 0.864       |
| time/                   |             |
|    fps                  | 514         |
|    iterations           | 2           |
|    time_elapsed         | 15          |
|    total_timesteps      | 8192        |
| train/                  |             |
|    approx_kl            | 0.025726061 |
|    clip_fraction        | 0.183       |
|    clip_range           | 0.2         |
|    entropy_loss         | -0.925      |
|    explained_variance   | 0.674       |
|    learning_rate        | 0.00025     |
|    loss                 | 0.0975    

KeyboardInterrupt: 

In [ ]:
# Plot training curves
import matplotlib.pyplot as plt
import numpy as np

all_rewards = callback.episode_rewards + callback2.episode_rewards
all_lengths = callback.episode_lengths + callback2.episode_lengths
all_flags   = callback.episode_flags   + callback2.episode_flags

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
window = min(100, len(all_rewards))

for ax, data, color, title in zip(
    axes,
    [all_rewards, all_lengths, None],
    ['blue', 'orange', 'green'],
    ['Episode Rewards', 'Episode Lengths', 'Cumulative Flag Rate'],
):
    if title == 'Cumulative Flag Rate':
        if all_flags:
            cum = np.cumsum(all_flags) / (np.arange(len(all_flags)) + 1)
            ax.plot(cum, color=color, linewidth=2)
            ax.set_ylim(0, 1)
    else:
        ax.plot(data, alpha=0.3, color=color)
        if len(data) > window:
            smoothed = np.convolve(data, np.ones(window) / window, mode='valid')
            ax.plot(range(window - 1, len(data)), smoothed, color=color, linewidth=2)
    ax.set_title(title)
    ax.set_xlabel('Episode')

plt.tight_layout()
plt.savefig('../models/symbolic_ppo/training_curves.png', dpi=150)
plt.show()

In [ ]:
# Evaluate final model — 10 episodes
import numpy as np
from stable_baselines3 import PPO

eval_model = PPO.load('../models/symbolic_ppo/final_model')

eval_env = make_symbolic_env(
    env_id='SuperMarioBros-1-1-v3',
    skip=4, n_stack=4, flatten=True,
)

rewards, lengths, flags = [], [], []

for ep in range(10):
    reset_result = eval_env.reset()
    obs = reset_result[0] if isinstance(reset_result, tuple) else reset_result
    done, total_reward, steps, flag = False, 0.0, 0, False

    while not done:
        action, _ = eval_model.predict(obs, deterministic=True)
        result = eval_env.step(int(action))
        if len(result) == 5:
            obs, reward, terminated, truncated, info = result
            done = terminated or truncated
        else:
            obs, reward, done, info = result
        total_reward += float(reward)
        steps += 1
        if isinstance(info, dict) and info.get('flag_get', False):
            flag = True

    rewards.append(total_reward)
    lengths.append(steps)
    flags.append(flag)
    print(f'Episode {ep+1:2d}: reward={total_reward:.1f}, steps={steps}, {"FLAG!" if flag else "DEAD"}')

print(f'\nMean reward: {np.mean(rewards):.1f} ± {np.std(rewards):.1f}')
print(f'Mean length: {np.mean(lengths):.0f}')
print(f'Flag rate:   {np.mean(flags):.0%}')

eval_env.close()

In [ ]:
# Watch: python watch_agent.py --model ../models/symbolic_ppo/final_model --env SuperMarioBros-1-1-v3